# 01 — Simple model

First-pass model of a hydrodynamically focused flow cytometer channel and the ADC sample rate it requires.

Workflow: derive/try things here, and once a formula stabilises move it into `src/cytosim/`.
`autoreload` picks up edits to the package without restarting the kernel.

**Units:** `Params` fields are stored in SI. Type lengths as `um(...)` and flow rates as `ul_per_s(...)` / `ml_per_min(...)`.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

from cytosim import Params, simulate, units
from cytosim.units import um, ul_per_s, ml_per_min
from cytosim.params import MILK_PARTICLES, MEASURED_CORE_WIDTH_RANGE
from cytosim import geometry, flow, optics, particle, signal


def report(r):
    print(f"core width      : {units.to_um(r.core_width):8.1f} µm   (measured {units.to_um(MEASURED_CORE_WIDTH_RANGE[0]):.0f}-{units.to_um(MEASURED_CORE_WIDTH_RANGE[1]):.0f} µm; plug-flow bound {units.to_um(r.core_diameter_plug):.1f} µm)")
    print(f"velocity        : {r.velocity_min:8.2f} - {r.velocity:.2f} m/s across the core   (bulk {r.velocity_bulk:.2f} m/s, centreline/bulk {r.velocity_ratio:.3f})")
    print(f"Reynolds        : {r.reynolds:8.0f}")
    print(f"pulse FWHM      : {r.fwhm*1e9:8.0f} - {r.fwhm_max*1e9:.0f} ns   (fastest - slowest particle)")
    print(f"pulse sigma_t   : {r.sigma_t*1e9:8.0f} ns")
    print(f"f_3dB           : {r.f_3db/1e6:8.2f} MHz")
    print(f"f_s (FWHM crit.): {r.f_s_fwhm/1e6:8.2f} MHz")
    print(f"f_s (BW crit.)  : {r.f_s_bandwidth/1e6:8.2f} MHz")
    print(f"f_s required    : {r.f_s_required/1e6:8.2f} MHz   (plug-flow lower bound {r.f_s_required_plug/1e6:.2f} MHz)")

## Instrument defaults

200 × 200 µm channel, sheath 20 mL/min, sample 10 µL/s (EB, 1:1) or 7 µL/s (PR2, 1:3.2), 20 × 100 µm spot.

In [ ]:
p = Params.eb()
p

In [ ]:
for name, preset in (("EB 1:1", Params.eb), ("PR2 1:3.2", Params.pr2)):
    print(f"=== {name} ===")
    report(simulate(preset()))
    print()

## Velocity field in the square duct

Fluid sticks to the walls (no-slip), so the velocity is zero at the wall and highest on the axis. A round pipe gives a parabola with centreline = 2 × mean; a **square duct** has no closed form, and `flow.RectangularDuct` uses the classic Fourier series (White, *Viscous Fluid Flow*, eq. 3.48): cosines across one axis times cosh terms across the other, each term already vanishing on the walls.

Checks: the mean over the section must be 1 (so it integrates to $Q_\text{total}$), and centreline/mean must be **2.096** for a square (1.5 in the parallel-plate limit).

In [ ]:
duct = flow.RectangularDuct(um(200), um(200))
plug = flow.PlugFlow(um(200), um(200))

n = 401
xs = np.linspace(-duct.width / 2, duct.width / 2, n)
ys = np.linspace(-duct.depth / 2, duct.depth / 2, n)
X, Y = np.meshgrid(xs, ys, indexing="ij")
G = duct.shape(X, Y)
print(f"centreline / mean : {duct.centreline_ratio:.4f}   (square duct literature: 2.096; round pipe: 2.0)")
print(f"mean over section : {np.trapezoid(np.trapezoid(G, ys, axis=1), xs) / (duct.width * duct.depth):.5f}   (must be 1)")
print(f"parallel plates   : {flow.RectangularDuct(um(200), um(200) * 100).centreline_ratio:.3f}   (limit 1.5)")
print(f"value at the wall : {duct.shape(duct.width / 2, 0):.1e}")

### The focused core

The sample is injected on the axis, where the fluid is fastest, and the sheath squeezes it until the flow *through the core region* equals $Q_\text{sample}$ (mass conservation, `geometry.core_size`). Because the core rides on the fast centre, it needs less area than the plug-flow estimate $A\,Q_\text{sample}/Q_\text{total}$ — that is why plug flow overestimates the width.

The core is assumed circular (`core_aspect = 1`); a ribbon core (thin across the beam, tall along it) is available via `core_aspect > 1` once the injector geometry is known.

In [ ]:
p = Params.eb()
r = simulate(p)
rx, ry = geometry.core_size(duct, p.q_sample, p.q_total, p.core_aspect)
V = duct.velocity(X, Y, p.q_total)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
im = ax1.pcolormesh(units.to_um(X), units.to_um(Y), V, shading="auto", cmap="viridis")
fig.colorbar(im, ax=ax1, label="velocity [m/s]")
th = np.linspace(0, 2 * np.pi, 200)
ax1.plot(units.to_um(rx * np.cos(th)), units.to_um(ry * np.sin(th)), "w-", lw=1.5, label=f"sample core, {units.to_um(r.core_width):.1f} µm")
ax1.set_aspect("equal"); ax1.set_xlabel("x, along the laser beam [µm]"); ax1.set_ylabel("y, across the beam [µm]")
ax1.set_title("laminar velocity field, 200 × 200 µm duct"); ax1.legend(loc="upper right")

ax2.plot(units.to_um(ys), duct.velocity(0.0, ys, p.q_total), label="square duct (series)")
ax2.plot(units.to_um(ys), duct.centreline_ratio * r.velocity_bulk * (1 - (2 * ys / duct.depth) ** 2), ":", label="parabola with the same peak")
ax2.axhline(r.velocity_bulk, color="k", ls="--", label=f"plug / bulk {r.velocity_bulk:.1f} m/s")
ax2.axvspan(-units.to_um(ry), units.to_um(ry), alpha=0.2, label="core")
ax2.set_xlabel("y [µm]"); ax2.set_ylabel("velocity [m/s]"); ax2.legend(); ax2.grid(alpha=0.3)
ax2.set_title("profile through the axis")
print(f"core width {units.to_um(r.core_width):.1f} µm (plug flow would give {units.to_um(r.core_diameter_plug):.1f} µm); "
      f"velocity across the core {r.velocity_min:.2f}-{r.velocity:.2f} m/s")

### Pulse-width distribution across the core

Particles at the core edge move slower than on the axis, so pulses have a spread of widths. The distribution is flux-weighted (`geometry.core_velocities`): a fluid element carries particles in proportion to $u\,dA$. The core is so small compared with the channel that it sits on the flat top of the profile — the spread is only a few percent, and the **fastest (centreline) particle** sets the sampling rate. Plug flow is reported as the lower bound because at Re ≈ 2500 the real profile is somewhat flatter than fully laminar.

In [ ]:
v, w = geometry.core_velocities(duct, p.q_total, rx, ry, n_r=200, n_theta=64)
# FWHM ∝ 1/v for a given particle: scale the centreline value
fwhm = r.fwhm * r.velocity / v

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.hist(v, bins=40, weights=w / w.sum(), color="C0")
ax1.axvline(r.velocity_bulk, color="k", ls="--", label="plug flow")
ax1.set_xlabel("particle velocity [m/s]"); ax1.set_ylabel("fraction of detected particles"); ax1.legend(); ax1.grid(alpha=0.3)
ax2.hist(fwhm * 1e9, bins=40, weights=w / w.sum(), color="C1")
ax2.set_xlabel(f"pulse FWHM [ns], {units.to_um(p.particle_diameter):.0f} µm particle"); ax2.set_ylabel("fraction of detected particles"); ax2.grid(alpha=0.3)
print(f"FWHM {r.fwhm*1e9:.0f} ns (fastest) to {r.fwhm_max*1e9:.0f} ns (slowest), spread {(r.fwhm_max / r.fwhm - 1) * 100:.1f} %")
print(f"f_s required {r.f_s_required/1e6:.2f} MHz from the fastest particle; plug-flow lower bound {r.f_s_required_plug/1e6:.2f} MHz")

## Detector pulse shape per milk particle type

Top-hat particle convolved with the Gaussian beam (20 µm along the flow).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for name, spec in MILK_PARTICLES.items():
    rr = simulate(Params(particle_diameter=spec["d"]))
    ax.plot(rr.t * 1e6, rr.pulse, label=f"{name} ({units.to_um(spec['d']):.1f} µm), FWHM = {rr.fwhm*1e9:.0f} ns")
rr = simulate(Params(particle_diameter=um(20)))
ax.plot(rr.t * 1e6, rr.pulse, "--", label=f"large fat globule (20 µm), FWHM = {rr.fwhm*1e9:.0f} ns")
ax.set_xlabel("time [µs]")
ax.set_ylabel("normalised signal")
ax.legend()
ax.grid(alpha=0.3)

## Sample rate vs sheath flow

In [ ]:
q_sheath = np.linspace(2, 40, 40)  # mL/min
res = [simulate(Params(q_sheath=ml_per_min(q))) for q in q_sheath]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(q_sheath, [r.f_s_required / 1e6 for r in res])
ax1.axvline(20, color="k", ls=":", label="instrument")
ax1.set_xlabel("sheath flow [mL/min]"); ax1.set_ylabel("required sample rate [MHz]"); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(q_sheath, [units.to_um(r.core_diameter) for r in res])
ax2.axhspan(*[units.to_um(x) for x in MEASURED_CORE_WIDTH_RANGE], alpha=0.15, label="measured 10-30 µm")
ax2.axvline(20, color="k", ls=":")
ax2.set_xlabel("sheath flow [mL/min]"); ax2.set_ylabel("core diameter [µm]"); ax2.legend(); ax2.grid(alpha=0.3)

## Scratch

Try new physics below; promote to the package when it settles.